In [1]:
import pandas as pd
import numpy as np
import statistics
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [2]:
df_org  = pd.read_csv(r"/home/ziyad/Desktop/IA_DATA_2026_2027/02_ISMO_COURS_ET_TP/Principale/Module_1_Math/projet_final/Analyse-des-commandes-et-des-messages-clients-de-MarocShop/commandes.csv")

In [3]:
df = df_org.copy()

In [4]:
df.head()

,id_commande,date_commande,ville,sexe,age_client,categorie,canal,nb_articles,montant_dh,entrepot,delai_livraison_jours,colis_endommage,satisfaction,retour_client
0,CMD0001,2025-01-01,Casablanca,Femme,44.0,Beauté,Application,3.0,210.97,A,2.0,Non,Moyen,Non
1,CMD0002,2025-01-01,Tétouan,Femme,26.0,Mode,Web,5.0,579.45,B,5.0,Non,Faible,Non
2,CMD0003,2025-01-03,Tanger,Homme,28.0,Sport,Application,2.0,372.35,B,2.0,Non,elevé,Non
3,CMD0004,2025-01-05,Rabat,Femme,18.0,Mode,Application,3.0,478.11,C,4.0,Oui,Faible,Oui
4,CMD0005,2025-01-05,Rabat,Femme,41.0,Électronique,Application,7.0,411.04,C,4.0,Non,Élevé,Non


In [5]:
df.dtypes

id_commande                  str
date_commande                str
ville                        str
sexe                         str
age_client               float64
categorie                    str
canal                        str
nb_articles              float64
montant_dh               float64
entrepot                     str
delai_livraison_jours    float64
colis_endommage              str
satisfaction                 str
retour_client                str
dtype: object

In [6]:
print(f' les nombre des doublans {df.duplicated().sum()}')

 les nombre des doublans 18


In [7]:
valeurs_num = df.select_dtypes(include= "number")
valeurs_cat = df.select_dtypes(include= "str")
for i in valeurs_num :
    manquante_num = ((valeurs_num[i].isnull().mean())*100).round(1)
    print(i ,":" , manquante_num , "%")
for j in valeurs_cat:
    manquante_cat = ((valeurs_cat[j].isnull().mean())*100).round(1)
    
    print(j ,":"  ,manquante_cat , "%")

age_client : 4.4 %
nb_articles : 1.2 %
montant_dh : 2.9 %
delai_livraison_jours : 3.5 %
id_commande : 0.0 %
date_commande : 0.0 %
ville : 0.0 %
sexe : 1.9 %
categorie : 0.0 %
canal : 1.5 %
entrepot : 0.0 %
colis_endommage : 0.0 %
satisfaction : 8.3 %
retour_client : 0.0 %


In [8]:
df.columns.str.strip().str.lower()

Index(['id_commande', 'date_commande', 'ville', 'sexe', 'age_client',
       'categorie', 'canal', 'nb_articles', 'montant_dh', 'entrepot',
       'delai_livraison_jours', 'colis_endommage', 'satisfaction',
       'retour_client'],
      dtype='str')

In [9]:
df['date_commande'] = pd.to_datetime(df['date_commande'] , format= 'mixed')

In [10]:
future = df['date_commande'] > pd.Timestamp.today()
future.sum()

np.int64(0)

In [11]:
print(df.duplicated(subset= 'id_commande').sum())

18


In [12]:
df.duplicated().sum()

np.int64(18)

In [13]:
df = df.drop_duplicates()

In [14]:
df.shape

(500, 14)

In [15]:
df = df.drop_duplicates(subset= 'id_commande')

In [16]:
df.shape

(500, 14)

In [17]:
df.describe()

,date_commande,age_client,nb_articles,montant_dh,delai_livraison_jours
count,500,478.000000,494.000000,485.000000,482.000000
mean,2025-07-02 15:07:12,34.941423,2.522267,710.774412,3.537344
min,2025-01-01 00:00:00,0.000000,0.000000,-150.000000,-2.000000
25%,2025-04-01 18:00:00,27.000000,1.000000,261.170000,2.000000
50%,2025-07-04 00:00:00,35.000000,2.000000,385.870000,3.000000
75%,2025-09-29 00:00:00,42.000000,3.000000,616.060000,4.000000
max,2025-12-30 00:00:00,130.000000,40.000000,32000.000000,60.000000
std,NaN,11.686359,2.620428,2227.797358,3.834364


In [18]:
df.isnull().sum().sort_values(ascending=False)

satisfaction             40
age_client               22
delai_livraison_jours    18
montant_dh               15
sexe                     10
canal                     8
nb_articles               6
ville                     0
id_commande               0
date_commande             0
categorie                 0
entrepot                  0
colis_endommage           0
retour_client             0
dtype: int64

In [19]:
df['satisfaction'] = df['satisfaction'].fillna(df['satisfaction'].mode()[0])

In [20]:
df['sexe'] = df['sexe'].fillna(df['sexe'].mode()[0])

In [21]:
df['ville'] = df['ville'].fillna(df['ville'].mode())

In [22]:
df['canal'] = df['canal'].fillna(df['canal'].mode()[0])

In [23]:
moyenne_age = np.mean(df['nb_articles'])
df['nb_articles'] = df['nb_articles'].fillna(moyenne_age)

In [24]:
moyenne_delai_livraison_jours = np.mean(df['delai_livraison_jours'])
df['delai_livraison_jours'] = df['delai_livraison_jours'].fillna(moyenne_delai_livraison_jours)

In [25]:
moyenne_montant = np.mean(df['montant_dh'])
df['montant_dh'] = df['montant_dh'].fillna(moyenne_montant)

In [26]:
moyenne_age  = np.mean(df['age_client'])
df['age_client'] = df['age_client'].fillna(moyenne_age)

In [27]:
df.isnull().sum()

id_commande              0
date_commande            0
ville                    0
sexe                     0
age_client               0
categorie                0
canal                    0
nb_articles              0
montant_dh               0
entrepot                 0
delai_livraison_jours    0
colis_endommage          0
satisfaction             0
retour_client            0
dtype: int64

In [28]:
for i in ["age_client", "nb_articles", "montant_dh", "delai_livraison_jours"] :
    q1, q3 = df[i].quantile([0.25, 0.75])
    iqr = q3 - q1
    masque = (df[i] < q1 - 1.5 * iqr) | (df[i] > q3 + 1.5 * iqr)
    print(i, masque.sum(), "outliers")
    df.loc[masque, i] = df[i].median()

age_client 3 outliers
nb_articles 8 outliers
montant_dh 40 outliers
delai_livraison_jours 4 outliers


In [29]:
df.describe()

,date_commande,age_client,nb_articles,montant_dh,delai_livraison_jours
count,500,500.000000,500.000000,500.000000,500.000000
mean,2025-07-02 15:07:12,34.655071,2.320267,428.839092,3.279344
min,2025-01-01 00:00:00,5.000000,0.000000,-150.000000,1.000000
25%,2025-04-01 18:00:00,27.000000,1.000000,266.042500,2.000000
50%,2025-07-04 00:00:00,34.941423,2.000000,390.867500,3.000000
75%,2025-09-29 00:00:00,41.000000,3.000000,538.317500,4.000000
max,2025-12-30 00:00:00,62.000000,6.000000,1200.600000,7.000000
std,NaN,9.802417,1.256128,246.068646,1.353523


In [30]:
df.describe()

,date_commande,age_client,nb_articles,montant_dh,delai_livraison_jours
count,500,500.000000,500.000000,500.000000,500.000000
mean,2025-07-02 15:07:12,34.655071,2.320267,428.839092,3.279344
min,2025-01-01 00:00:00,5.000000,0.000000,-150.000000,1.000000
25%,2025-04-01 18:00:00,27.000000,1.000000,266.042500,2.000000
50%,2025-07-04 00:00:00,34.941423,2.000000,390.867500,3.000000
75%,2025-09-29 00:00:00,41.000000,3.000000,538.317500,4.000000
max,2025-12-30 00:00:00,62.000000,6.000000,1200.600000,7.000000
std,NaN,9.802417,1.256128,246.068646,1.353523
